# Faz 3 v2 — HFP Grafting: Qwen2.5-1.5B + O(1) Bellek (Distilasyon)

> **v2:** PYTHONPATH, teacher-sanity kiyasi ve HF label cift-kaydirma hatalari duzeltildi; Qwen varsayilan; Stage 2 gradient checkpointing. Eski `colab_graft_llama1b.ipynb` yerine BUNU kullanin.

`LLM_GRAFTING_STRATEGY.md` + `hfp/models/grafting.py` implementasyonunun eğitim/validasyon hattı.

**Akış:**
1. Ortam + `graft_smoke.py` (torch ortamında ilk kez — GEÇMELİ)
2. Llama-3.2-1B yükle → graft öncesi referans PPL
3. Graft (katmanların yarısı HFP: kafa-başına bellek, RoPE bypass, warm-start) → **Sıfırıncı-adım sanity** (strateji dok. §4.1: PPL çöp olmamalı)
4. **Stage 1:** teacher-forcing distilasyon (katman-içi MSE; öğretmen-kopyasız → tek T4'e sığar)
5. **Stage 2:** logit-KL + LM loss (uçtan uca, student modu)
6. **Validasyon:** kısa-bağlam PPL kaybı (<%5 hedef), iğne-deliği (needle) streaming testi, VRAM sabitliği

**Önceden yazılı başarı kriterleri (strateji dok. §4):**
- Zero-shot: graft sonrası PPL < 1000 (ağırlık transferi doğru).
- Stage 2 sonrası kısa-bağlam PPL, orijinalin ≤1.05×'i.
- Needle: HFP-Llama, KV-cache'siz streaming'de 8K+ geriden iğneyi bulmalı.
- VRAM: 10.000. tokende grafted katman state boyutu 1. token ile aynı.

> **Donanım:** T4 (ücretsiz Colab) yeter — fp32, batch 1, grad-accum. A100/L4 varsa `BATCH`/`SEQ` büyütün.
> **Token güvenliği:** HF token'ı Colab Secrets'a (sol panel 🔑, ad: `HF_TOKEN`) ekleyin ya da sorulduğunda girin. ASLA koda/nota yazmayın.

In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys

BASE = '/content' if os.path.exists('/content') else ('/kaggle/working' if os.path.exists('/kaggle') else '.')
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers>=4.46', 'datasets', 'accelerate'], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)

# HF girisi: once Colab Secrets, yoksa gizli prompt (token koda yazilmaz!)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token Colab Secrets\'tan alindi.')
except Exception:
    if not os.environ.get('HF_TOKEN'):
        from getpass import getpass
        os.environ['HF_TOKEN'] = getpass('HF token: ')

import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0) if DEV=="cuda" else "CPU"}')

# graft_smoke: torch ortaminda ILK kosum — gecmeden devam etmeyin
env = dict(os.environ, PYTHONPATH=REPO + ':' + os.environ.get('PYTHONPATH', ''))
r = subprocess.run([sys.executable, 'review_scripts/graft_smoke.py'],
                   capture_output=True, text=True, env=env, cwd=REPO)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'graft_smoke.py BASARISIZ — devam etmeyin'

In [ ]:
# --- 2. MODEL + GRAFT ONCESI REFERANS PPL ---
import math, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL_ID = 'Qwen/Qwen2.5-1.5B'           # ungated, Apache 2.0
# MODEL_ID = 'meta-llama/Llama-3.2-1B'   # gated alternatif (HF'de lisans kabulu gerekir)

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32,
                                             token=os.environ['HF_TOKEN']).to(DEV)
model.eval()
print(f'{MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B param')

wiki = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='validation')
val_text = '\n'.join(t for t in wiki['text'] if t.strip())
val_ids = tok(val_text, return_tensors='pt').input_ids[0]
print(f'Val: {len(val_ids):,} token')

@torch.no_grad()
def ppl(model, ids, seq=1024, n_chunks=24):
    model.eval(); losses = []
    for i in range(n_chunks):
        s = i * seq
        if s + seq > len(ids): break
        x = ids[s:s+seq].unsqueeze(0).to(DEV)
        out = model(x, labels=x)   # HF labels'i iceride kaydirir; disaridan kaydirma = cift-kaydirma hatasi
        losses.append(out.loss.item())
    return math.exp(sum(losses)/len(losses))

PPL_BASE = ppl(model, val_ids)
print(f'GRAFT ONCESI referans PPL: {PPL_BASE:.2f}')

In [ ]:
# --- 3. GRAFT + SIFIRINCI-ADIM SANITY ---
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 distill_loss, trainable_parameters,
                                 enable_streaming, reset_streaming, HFPGraftAttention)

gcfg = GraftConfig(decay_mode='cubic_flux_chunked', write_rule='hybrid',
                   key_feature_map='dpfp', rec_block=64)
GRAFT_LAYERS = graft_llama(model, gcfg)      # default: tek-indeksli katmanlar (yari-yariya hibrit)

# teacher modu == orijinal model (dogrulama) — AYNI chunk sayisiyla kiyasla!
set_graft_mode(model, 'teacher')
ppl_teacher = ppl(model, val_ids)            # PPL_BASE ile ayni 24 chunk
print(f'teacher {ppl_teacher:.2f} vs base {PPL_BASE:.2f}')
assert abs(ppl_teacher - PPL_BASE) < PPL_BASE*0.01, 'teacher yolu bozuk!'

# student modu, egitimsiz: PPL kotu olacak ama COP OLMAMALI (kriter: <1000)
set_graft_mode(model, 'student')
PPL_ZERO = ppl(model, val_ids, n_chunks=8)
print(f'Zero-shot (egitimsiz HFP) PPL: {PPL_ZERO:.1f}  (kriter <1000: {"GECTI" if PPL_ZERO < 1000 else "KALDI — agirlik transferi/olcek kontrol"})')

In [ ]:
# --- 4. STAGE 1: TEACHER-FORCING DISTILASYON (katman-ici MSE) ---
# Ileriye ogretmen ciktisi gider -> katmanlar bagimsiz, akis on-distribution.
# Egitilen yalniz HFP parametreleri (iki-LR kurali burada tek grup: hepsi HFP).
import time
from datasets import load_dataset
from transformers import get_cosine_schedule_with_warmup

SEQ, BATCH, STEPS1, LR1 = 1024, 1, 2000, 1e-3
ACCUM = 4

train_stream = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1',
                            split='train', streaming=True)

def batch_iter(stream, seq, batch):
    buf = []
    for ex in stream:
        if not ex['text'].strip(): continue
        buf.extend(tok(ex['text']).input_ids)
        while len(buf) >= batch * (seq + 1):
            chunk = buf[:batch*(seq+1)]; buf = buf[batch*(seq+1):]
            t = torch.tensor(chunk).view(batch, seq+1)
            yield t[:, :-1], t[:, 1:]

model.config.use_cache = False   # egitimde cache kapali
opt = torch.optim.AdamW(trainable_parameters(model), lr=LR1, weight_decay=0.01)
sch = get_cosine_schedule_with_warmup(opt, 100, STEPS1)
set_graft_mode(model, 'teacher_forcing')
model.train()

it = batch_iter(train_stream, SEQ, BATCH)
t0 = time.time(); log = []
for step in range(1, STEPS1 + 1):
    loss_acc = 0.0
    for _ in range(ACCUM):
        x, _ = next(it)
        model(x.to(DEV))
        aux = distill_loss(model)
        (aux / ACCUM).backward()
        loss_acc += aux.item() / ACCUM
    torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
    opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    if step % 50 == 0:
        alphas = [torch.sigmoid(m.alpha_logit).mean().item()
                  for m in model.modules() if isinstance(m, HFPGraftAttention)]
        print(f'S1 {step}/{STEPS1}  mse {loss_acc:.5f}  alpha_ort {sum(alphas)/len(alphas):.3f}  {(time.time()-t0)/60:.0f}dk', flush=True)
        log.append((step, loss_acc))
    if step % 500 == 0:
        torch.save({n: p for n, p in model.state_dict().items()
                    if any(k in n for k in ('decay','log_eta','conv_q','conv_k','beta_gate','alpha_logit','retrieval_norm','out_gain'))},
                   f'{BASE}/hfp_graft_stage1_{step}.pt')
print('Stage 1 tamam.')

In [ ]:
# --- 5. STAGE 2: LOGIT-KL + LM LOSS (uctan uca) ---
# Ogretmen logitleri ayni modelin "teacher" modundan (ikinci kopya yok).
import torch.nn.functional as F

STEPS2, LR2, KL_W, TEMP = 1000, 3e-4, 0.5, 2.0
# [VRAM] Qwen vocab 151k -> logit tensorleri buyuk; checkpointing T4 icin sart
model.gradient_checkpointing_enable()
model.config.use_cache = False
opt = torch.optim.AdamW(trainable_parameters(model), lr=LR2, weight_decay=0.01)
sch = get_cosine_schedule_with_warmup(opt, 50, STEPS2)

for step in range(1, STEPS2 + 1):
    loss_acc = 0.0
    for _ in range(ACCUM):
        x, y = next(it)
        x, y = x.to(DEV), y.to(DEV)
        with torch.no_grad():
            set_graft_mode(model, 'teacher')
            t_logits = model(x).logits
        set_graft_mode(model, 'student')
        out = model(x, labels=x)   # HF ic kaydirma
        kl = F.kl_div(F.log_softmax(out.logits/TEMP, -1),
                      F.log_softmax(t_logits/TEMP, -1),
                      log_target=True, reduction='batchmean') * TEMP * TEMP
        loss = out.loss + KL_W * kl
        (loss / ACCUM).backward()
        loss_acc += loss.item() / ACCUM
    torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
    opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    if step % 50 == 0:
        print(f'S2 {step}/{STEPS2}  loss {loss_acc:.4f}', flush=True)
torch.save({n: p for n, p in model.state_dict().items()
            if any(k in n for k in ('decay','log_eta','conv_q','conv_k','beta_gate','alpha_logit','retrieval_norm','out_gain'))},
           f'{BASE}/hfp_graft_final.pt')
print('Stage 2 tamam; HFP parametreleri kaydedildi.')

In [ ]:
# --- 6. VALIDASYON: PPL + NEEDLE + VRAM ---
import math, random, torch

model.gradient_checkpointing_disable()
model.config.use_cache = True
# 6a. Kisa-baglam PPL kaybi (kriter: <= 1.05x orijinal)
set_graft_mode(model, 'student')
PPL_FINAL = ppl(model, val_ids)
print(f'PPL: orijinal {PPL_BASE:.2f} -> HFP-hibrit {PPL_FINAL:.2f} '
      f'({PPL_FINAL/PPL_BASE:.3f}x; kriter <=1.05x: {"GECTI" if PPL_FINAL <= 1.05*PPL_BASE else "KALDI"})')

# 6b. Igne-deligi (needle) — O(1) streaming: grafted state tasinir,
# full-attn katmanlar icin HF cache kullanilir.
from transformers import DynamicCache

def needle_test(model, tok, total_tokens=8192, chunk=512, seed=0):
    random.seed(seed)
    secret = random.choice(['blue falcon', 'copper mountain', 'silent river'])
    needle = f' The secret passphrase is {secret}. '
    filler = ' The weather was ordinary and nothing of note happened that day.'
    fill_ids = tok(filler, add_special_tokens=False).input_ids
    text_ids = []
    while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
    ins = len(text_ids) // 8                      # igne basa yakin -> uzun gap
    needle_ids = tok(needle, add_special_tokens=False).input_ids
    text_ids = text_ids[:ins] + needle_ids + text_ids[ins:total_tokens - len(needle_ids)]
    query_ids = tok(' The secret passphrase is', add_special_tokens=False).input_ids
    ids = torch.tensor(text_ids + query_ids).unsqueeze(0).to(DEV)

    set_graft_mode(model, 'student'); model.eval()
    enable_streaming(model, True); reset_streaming(model)
    cache = DynamicCache()
    with torch.no_grad():
        for s in range(0, ids.size(1), chunk):
            out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
            cache = out.past_key_values
        gen = []
        last = out.logits[:, -1:].argmax(-1)
        for _ in range(6):
            gen.append(last.item())
            out = model(last, past_key_values=cache, use_cache=True)
            cache = out.past_key_values
            last = out.logits[:, -1:].argmax(-1)
    enable_streaming(model, False)
    answer = tok.decode(gen)
    hit = secret.split()[0] in answer
    print(f'  L={total_tokens}: gizli="{secret}" cevap="{answer.strip()}" -> {"BULDU" if hit else "BULAMADI"}')
    return hit

print('Needle (igne) testi:')
for L in [2048, 8192, 16384]:
    needle_test(model, tok, total_tokens=L)

# 6c. VRAM / state sabitligi: grafted katman state boyutu tokenle degismemeli
attn = next(m for m in model.modules() if isinstance(m, HFPGraftAttention))
if attn._stream_state is not None:
    M, z = attn._stream_state[0], attn._stream_state[1]
    print(f'Grafted state: M {tuple(M.shape)} + z {tuple(z.shape)} '
          f'= {(M.numel()+z.numel())*4/1e6:.1f} MB — baglam uzunlugundan BAGIMSIZ (O(1))')
if DEV == 'cuda':
    print(f'Tepe VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
print('\nBitince: GPU_ROADMAP.md §10 checklist + RESULTS.md guncellenecek.')